# Looking at the data before building anything

The step that gets skipped, done properly once: class balance, duplicates, leakage candidates, and a common-sense baseline — all before a single layer is written.

**Runs on:** CPU — about 2 minutes &nbsp;·&nbsp; **Slides:** [Chapter 6 — The Universal Workflow of Machine Learning](../../../course-web-slides/ch06/index.html) &nbsp;·&nbsp; **Section:** 01 — Define the task

---

## The dataset

In [ ]:
import numpy as np
import pandas as pd
from keras.datasets import reuters

(train_data, train_labels), (test_data, test_labels) = reuters.load_data(
    num_words=10000)

print(f"train {len(train_data)}   test {len(test_data)}")
print(f"classes: {len(set(train_labels))}")
print(f"sequence length: min {min(map(len, train_data))}, "
      f"median {int(np.median([len(s) for s in train_data]))}, "
      f"max {max(map(len, train_data))}")

## Class balance

In [ ]:
import matplotlib.pyplot as plt

counts = np.bincount(train_labels)
order = counts.argsort()[::-1]

plt.figure(figsize=(10, 3.4))
plt.bar(range(len(counts)), counts[order])
plt.xlabel("class (sorted by frequency)"); plt.ylabel("samples")
plt.title("Severely imbalanced — the top class is a third of the data")
plt.show()

print(f"largest class: {counts.max()} samples ({counts.max()/len(train_labels):.1%})")
print(f"smallest class: {counts.min()} samples")
print(f"classes with fewer than 20 samples: {(counts < 20).sum()}")

**This changes the metric.** Plain accuracy on a dataset where one class is 36% of the data rewards a model that ignores the tail entirely. Chapter 6's rule — choose a measure of success that reflects what you actually want — starts here, not after training.

## Duplicates and near-duplicates

In [ ]:
as_tuples = [tuple(s) for s in train_data]
unique = len(set(as_tuples))
print(f"{len(as_tuples) - unique} exact duplicates in the training set")

# Also check across the train/test boundary -- the leak that matters.
train_set = set(as_tuples)
cross = sum(1 for s in test_data if tuple(s) in train_set)
print(f"{cross} test samples appear verbatim in training")

> ⚠️ **Any nonzero number on the second line invalidates the test score.** Deduplicate across the split, not just within it.

## Sequence length, and what it implies for preprocessing

In [ ]:
lengths = np.array([len(s) for s in train_data])

plt.figure(figsize=(7, 3.6))
plt.hist(lengths, bins=80)
for q in [0.5, 0.9, 0.99]:
    v = np.quantile(lengths, q)
    plt.axvline(v, ls="--", lw=1, label=f"{q:.0%} at {int(v)}")
plt.xlabel("tokens"); plt.legend(); plt.title("Sequence length distribution")
plt.show()

Truncating at the median throws away half of half the documents. Truncating at the 99th percentile pads almost everything. **Neither is free**, and the histogram is what turns that into a decision rather than a default.

## Three baselines, in ascending order of effort

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

def vectorize(seqs, dim=10000):
    out = np.zeros((len(seqs), dim), dtype="float32")
    for i, s in enumerate(seqs):
        out[i, s] = 1.
    return out

x_tr, x_te = vectorize(train_data), vectorize(test_data)

d = DummyClassifier(strategy="most_frequent").fit(x_tr, train_labels)
print(f"1. majority class:        {d.score(x_te, test_labels):.4f}")

lr = LogisticRegression(max_iter=400, n_jobs=-1).fit(x_tr, train_labels)
print(f"2. logistic on bag-of-words: {lr.score(x_te, test_labels):.4f}")

Expected output:

```
1. majority class:        0.36x
2. logistic on bag-of-words: 0.7xx
```

The second number is the real bar. **A deep model that scores 0.75 here has not earned its place** — and finding that out took thirty seconds rather than a week.

## What could leak

A checklist to run before every project, not only this one.

| Question | Why it matters |
|---|---|
| Is any feature computed **after** the label is known? | The classic target leak — a field that only exists once the outcome has happened. |
| Is the data ordered in time? | Then the split must be too. |
| Are samples grouped — by patient, customer, document? | The split must be by **group**, not by row. |
| Was anything fitted before the split? | Scalers, vocabularies, and `adapt()` calls all count. |
| Are there duplicates? | Checked above. |

None of these produce an error. All of them produce a good number.

---

## What to take away

- Class balance decides the metric, before any model exists.
- Check for duplicates **across** the split, not just within it.
- The length distribution turns truncation from a default into a decision.
- The baseline to beat is a simple model on the same features, not random guessing.